# M7 — QLoRA fine-tune of Qwen2.5-3B-Instruct

Run this on Colab (T4) or Kaggle (2xT4) -- NOT on the local RTX 5050, since bitsandbytes has no working Blackwell (sm_120) build. See `docs/PROJECT_PLAN.md`, "GGUF via llama.cpp, never bitsandbytes".

Hyperparameters per the project plan: r=16, alpha=32, 4-bit NF4 base.

**Before running:** upload `coaching_pairs.jsonl` (from `ml/finetuning/generate_coaching_pairs.py`, run locally) using the file picker in the left sidebar, or the upload cell below.

**If the install cell's bitsandbytes assertion fails:** go to Runtime > Change runtime type, confirm T4 GPU is selected, then Runtime > Restart session and rerun the install cell. This notebook intentionally installs unpinned/current package versions rather than the exact ones pinned in `ml/requirements.txt` (that repo's dev environment), since a pinned `bitsandbytes==0.45.0` had no CUDA binary for a recent Colab image and dragged in an incompatible triton.

In [ ]:
# Unpinned on purpose: exact versions pinned to this repo's local dev
# environment (transformers==4.47.1 etc) broke on Colab's current image --
# bitsandbytes had no matching CUDA binary and pulled in a triton version
# missing triton.ops. Let pip resolve current, mutually-compatible builds
# for whatever Python/CUDA Colab is actually running, and force a clean
# reinstall of the GPU-sensitive packages so cached wheels can't linger.
!pip install -q -U transformers peft trl accelerate datasets
!pip install -q -U --force-reinstall --no-cache-dir bitsandbytes

# Sanity check before spending time downloading the model: fail fast and
# loud if bitsandbytes still can't see a CUDA binary, rather than silently
# falling back to a CPU path that would make 4-bit loading fail later with
# a much more confusing error.
import bitsandbytes as bnb
assert bnb.COMPILED_WITH_CUDA, (
    "bitsandbytes has no CUDA binary for this runtime -- check Runtime > "
    "Change runtime type is set to a GPU (T4), then Runtime > Restart session "
    "and rerun this cell."
)
print(f"bitsandbytes {bnb.__version__}, CUDA build: {bnb.COMPILED_WITH_CUDA}")

In [ ]:
# Upload coaching_pairs.jsonl if not already present in the Colab filesystem.
import os
if not os.path.exists("coaching_pairs.jsonl"):
    from google.colab import files
    uploaded = files.upload()
    assert "coaching_pairs.jsonl" in uploaded, "Upload coaching_pairs.jsonl (produced by generate_coaching_pairs.py)"

In [ ]:
import json

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "qwen2.5-3b-coaching-lora"

SYSTEM_PROMPT = (
    "You are a warm, practical speaking-confidence coach for people with speech "
    "differences. You are not a clinician; never use diagnostic or treatment language. "
    "Give short, encouraging, concrete advice."
)

In [ ]:
pairs = []
with open("coaching_pairs.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            pairs.append(json.loads(line))

print(f"Loaded {len(pairs)} coaching pairs")
print(pairs[0])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

def to_chat_text(pair):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": pair["instruction"]},
        {"role": "assistant", "content": pair["response"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

texts = [to_chat_text(p) for p in pairs]
dataset = Dataset.from_dict({"text": texts})
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(dataset)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=1024,
)

trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

In [ ]:
# Zip and download the adapter for M8 (merge -> GGUF -> local llama.cpp).
import shutil
shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)

from google.colab import files
files.download(f"{OUTPUT_DIR}.zip")